# Evaluacion Parcial 1: Agente Inteligente de Soporte TI (SupportAi)
--------------------------------------------------------------------
# Presentacion del Caso y Contexto de nuestra Empresa

Nosotros como equipo hemos diseñado un proyecto orientado a resolver un cuello de botella muy comun en el mundo real: **la saturacion de las mesas de ayuda de TI**.

Para este proyecto, creamos la empresa **SupportAi**, basandonos en la operacion de sistemas de tickets reales como **Jira Service Desk**. En nuestra organizacion, el equipo de soporte de Nivel 1 esta abrumado. Diariamente, los tecnicos invierten mas del 50% de su tiempo respondiendo manualmente las mismas 6 preguntas:
1. ¿Como cambio mi contraseña?
2. ¿Como configuro el correo institucional?
3. ¿Que hago si se bloqueo mi cuenta?
4. ¿Como solicito acceso a una aplicacion?
5. ¿Como conecto el computador al Wi-Fi?
6. ¿Cual es el procedimiento para reportar un problema?

# 2. Problema y Solucion
Tambien detectamos que no podiamos simplemente conectar un LLM tradicional. Si lo haciamos, la IA podria **alucinar** y darle al trabajador instrucciones que no corresponden a las reglas de nuestra compañia (ej. inventar que la contraseña tenga 4 numeros, cuando por regla exigimos 12 caracteres).

Por ello, diseñamos una solucion basada en una arquitectura **RAG (Generacion Aumentada de Recuperacion)**.
El flujo metodologico que creamos es:
1. **Recuperacion (Retrieval):** Cuando el usuario hace la pregunta, no se la pasamos de inmediato a la IA. Primero, buscamos en nuestra base de datos el manual que resuelve esa duda especifica.
2. **Generacion:** Luego, le pasamos ese manual al modelo (Groq) y le obligamos a que redacte una respuesta amigable **basandose unicamente** en nuestra documentacion oficial.

In [13]:
# Librerias
!pip install -q openai faiss-cpu sentence-transformers matplotlib pandas numpy

In [14]:
# Conexion con LLM (GROQ)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from google.colab import userdata

# el cerebro GROQ
api_key = userdata.get("GROQ_API_KEY")
if not api_key:
    print(" ADVERTENCIA: No se encontro la API KEY.")
else:
    # Creamos el cliente de IA
    client = OpenAI(api_key=api_key, base_url="https://api.groq.com/openai/v1")
    print(" Cliente LLM configurado correctamente con Groq.")

 Cliente LLM configurado correctamente con Groq.


## 3. Arquitectura de Datos y Motor RAG

Cargamos nuestro xlsx donde esta la Base de Datos

### **¿Porque esta Arquitectura?:**
Para el modulo de busqueda usamos **FAISS**.
Cuando un empleado realiza una consulta, el sistema convierte su pregunta en un vector y lo compara con los vectores correspondientes a los manuales. De esta manera, podemos identificar matematicamente que contenido es mas similar o relevante para responder a la consulta.

In [15]:
# Excel
from google.colab import files
df = pd.read_excel("SupportAi_BaseConocimiento.xlsx")

print("Base de conocimiento cargada correctamente.")
print("Cantidad de procedimientos:", len(df))
display(df.head(7))

Base de conocimiento cargada correctamente.
Cantidad de procedimientos: 18


,ID,Categoría,Título,Procedimiento,Palabras clave,Área responsable,Prioridad,Canal/Sistema,SLA/Referencia,Estado
0,1,Seguridad,Cambio de contraseña,"Para cambiar la contraseña, ingrese al portal ...",contraseña;clave;renovar;cambiar contraseña;pa...,Seguridad TI,Media,MySupport,Cada 90 días,Activo
1,2,Correo,Configuración de correo institucional,"Abra Microsoft Outlook, seleccione 'Agregar cu...",correo;Outlook;Office 365;MFA;correo instituci...,Soporte TI,Media,Outlook / Authenticator,Según necesidad,Activo
2,3,Seguridad,Cuenta bloqueada,Si su cuenta fue bloqueada por intentos fallid...,cuenta bloqueada;bloqueo;intentos fallidos;des...,Seguridad TI,Alta,Anexo 5050 / WhatsApp soporte,Inmediato,Activo
3,4,Aplicaciones,Solicitud de acceso a aplicaciones,"Para solicitar acceso a un nuevo software, por...",SAP;Adobe;Figma;Canva;software;aplicación;Acce...,Soporte TI,Media,Jira Service Desk,Según necesidad,Activo
4,5,Redes,Conexión Wi-Fi corporativa,"Estando en la oficina, busque la red inalámbri...",wifi;Wi-Fi;red;internet;SupportAi_Corp_5G;cone...,Redes,Alta,SupportAi_Corp_5G,Según necesidad,Activo
5,6,Incidentes,Reportar un problema general,"Ante cualquier falla física del equipo, caída ...",problema;error;internet;pantalla azul;falla;in...,Mesa de Ayuda,Alta,Jira Service Desk,Menos de 2 horas,Activo
6,7,Hardware,Equipo no enciende,"Si el computador no enciende, no intente abrir...",computador;PC;no enciende;hardware;equipo;falla,Hardware,Alta,Jira Service Desk,Menos de 2 horas,Activo


In [16]:
documentos_internos = (df["Título"] + ": " + df["Procedimiento"]).tolist()

# 2. Motor de búsqueda FAISS
modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

# Convertimos los documentos en vectores
vectores = modelo_embeddings.encode(documentos_internos)
print("Embeddings creados correctamente")

dimension = vectores.shape[1]
indice_faiss = faiss.IndexFlatL2(dimension)
indice_faiss.add(vectores.astype("float32"))

print("Motor RAG inicializado correctamente")
print("Documentos Encontrados:", indice_faiss.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings creados correctamente
Motor RAG inicializado correctamente
Documentos Encontrados: 18


## 4. Diseño y Formulacion del Prompt

Para el diseño del prompt, necesitabamos garantizar que la IA hablara en tono profesional y que no inventara cosas.

**Justificacion System Prompt:**
Creamos una **REGLA CRITICA**. Le indicamos explicitamente al modelo que, si la informacion de FAISS no contiene una respuesta para la consulta del usuario, el agente debe reconocer que no tiene la informacion suficiente y lo llevaria a la mesa de ayuda humana.

Con esto evitamos que nuestra **Inteligencia** Mienta y que no ocacione otro problema.

In [17]:
# RAG
def agente_soporte_ti(pregunta):
    vector_pregunta = modelo_embeddings.encode(
    [pregunta]).astype("float32")

    # Buscar los 3 documentos más parecidos
    distancias, indices = indice_faiss.search(
    vector_pregunta,3)

    documentos = []
    for indice in indices[0]:
        documentos.append(documentos_internos[indice])

    contexto = "\n---\n".join(documentos)

    # Instrucciones
    prompt = """
    Eres SupportAi, un asistente de soporte TI.

    Responde solamente utilizando la información
    entregada en el contexto.

    No inventes información.

    Si la respuesta no aparece en el contexto,
    responde:

    "Lo siento, no tengo informacion oficial sobre
    este procedimiento. Por favor contacta a la
    mesa de ayuda."

    Responde de forma clara y amable.
    """

    mensaje = f"""Contexto:
    {contexto}
    Pregunta:
    {pregunta}
    """
    # Enviar pregunta a Groq
    respuesta = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            { "role": "system",
              "content": prompt
            },
            { "role": "user",
              "content": mensaje
            }],temperature=0.2)

    return respuesta.choices[0].message.content, contexto

## 5. Evaluacion de Coherencia y Resultados Reales

En esta prueba demostramos la coherencia de nuestro sistema. Simularemos ser 3 empleados haciendo preguntas de soporte. Vamos a imprimir en pantalla tanto el **Manual que encontro el RAG** como la **Respuesta final del Bot**, para demostrar que el modelo respetó fielmente los datos que nosotros le dimos.

In [18]:
# Chat SupportAi
print(" SupportAi - SOPORTE TI ")

print("Escribe tu problema de soporte TI.")
print("Escribe 'salir' para terminar.")

while True:
    pregunta = input("Tú: ")
    if pregunta.lower() == "salir":
        print("SupportAi: ¡Hasta luego!")
        break

    respuesta, documento = agente_soporte_ti(pregunta)
    print("SupportAi:")
    print(respuesta)


 SupportAi - SOPORTE TI 
Escribe tu problema de soporte TI.
Escribe 'salir' para terminar.
Tú: wifi
SupportAi:
Para conectarte a la red Wi‑Fi corporativa sigue estos pasos:

1. **Selecciona la red**  
   Busca y elige la red inalámbrica llamada **`SupportAi_Corp_5G`**.

2. **Introduce tus credenciales**  
   Cuando se te solicite usuario y contraseña, utiliza **las mismas credenciales con las que inicias sesión en tu computador Windows**.

3. **Verifica la conexión**  
   Si la conexión no se establece o sigue fallando, crea un ticket de soporte bajo el tipo **`Incidente TI`** (puedes hacerlo en Jira Service Desk o el sistema de tickets que utilicen).

¡Eso es todo! Si necesitas más ayuda, no dudes en abrir un ticket.
Tú: ayuda
SupportAi:
¡Hola! Para poder ayudarte de la mejor manera, ¿podrías describir brevemente el problema que estás experimentando? Si se trata de un error al iniciar sesión, de un fallo físico del equipo, de caída de internet o de una pantalla azul, dime cuál es y te

## 6. Dashboard y Conclusiones del Proyecto

Como equipo, programamos este dashboard usando `matplotlib` para evaluar el impacto de nuestra solucion en el negocio. Este grafico de barras ilustra y justifica por que este proyecto es util: logramos reducir los tiempos de resolucion del nivel 1 de soporte de minutos a milisegundos, justificando con creces la implementacion de esta arquitectura.

In [ ]:
categorias = ['Cambio Clave', 'Correo/Outlook', 'Cuenta Bloqueada', 'Acceso Apps', 'Wi-Fi', 'Otros Errores']
tiempo_humano_minutos = [15, 20, 30, 45, 10, 60]      # Tiempo del humano
tiempo_agente_minutos = [0.1, 0.1, 0.1, 0.1, 0.1, 2] # Tiempo de nuestro modelo IA

plt.figure(figsize=(10, 6))
x = np.arange(len(categorias))
width = 0.35

plt.bar(x - width/2, tiempo_humano_minutos, width, label='Soporte Humano (Minutos)', color='#FF6B6B')
plt.bar(x + width/2, tiempo_agente_minutos, width, label='Agente SupportAi (Minutos)', color='#4ECDC4')

plt.ylabel('Tiempo de Resolucion Promedio (Minutos)')
plt.title('Impacto del Proyecto: Tiempos de Respuesta - Humanos vs Nuestro Agente RAG')
plt.xticks(x, categorias, rotation=25)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.text(2.5, 50, " El Agente reduce los tiempos\nde espera en un 99.5%",
         fontsize=12, bbox=dict(facecolor='white', alpha=0.8, edgecolor='#4ECDC4'))

plt.tight_layout()
plt.show()

print(" Grafico listo.")